In [1]:
from pydantic_ai import Tool
from pydantic_ai.hooks.agent import ReactiveAgent

In [2]:
# Create a Simple Tool
async def greet(name: str) -> str:
    """Greet someone."""
    return f"Hello, {name}!"

tool = Tool(
    name="greet",
    function=greet,
    description="Greets a person"
)

In [3]:
# Create the model with OpenRouter configuration
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="env/.env")

api_key = os.environ.get("OPENROUTER_API_KEY")
base_url = os.environ.get("OPENROUTER_BASE_URL")

model = OpenAIModel(
    model_name="gpt-3.5-turbo",
    provider=OpenAIProvider(base_url=base_url, api_key=api_key),
)

In [4]:
# Create ReactiveAgent
agent = ReactiveAgent(
    model=model,
    tools=[tool],
    instructions="You are German and respond in German."
)

In [5]:
# Watch Model Changes
def on_model_change(change):
    print(f"Model changed from {change.old} to {change.new}")

# Watch the model value changes
agent.on.model.observe(on_model_change, names='value')

In [6]:
# Watch Instruction Changes
def on_instructions_change(change):
    print(f"Instructions changed from {change.old} to {change.new}")

# Watch the instructions value changes
agent.on.instructions.observe(on_instructions_change, names='value')

In [7]:
# Add Lifecycle Hooks
async def on_start(context):
    print(f"Agent starting with args: {context['args']}")

async def on_end(context):
    print("Agent finished")

# Current hook assignment
agent.on.start = on_start
agent.on.end = on_end

In [8]:
# Add Global Tool Hooks
async def before_any_tool(context):
    name = context['name']
    args = context['args']
    print(f"About to run tool: {name} with args: {args}")

async def after_any_tool(context):
    name = context['name']
    result = context['result']
    print(f"Tool {name} completed with result: {result}")

agent.on.before_any_tool = before_any_tool
agent.on.after_any_tool = after_any_tool

In [9]:
# Agent run
await agent.run("Please greet Alice")

Agent starting with args: ('Please greet Alice',)
About to run tool: greet with args: {"name":"Alice"}
Tool greet completed with result: Hello, Alice!
Agent finished


'Hello, Alice!'

In [10]:
# Change Model
new_model = OpenAIModel(
    model_name="gpt-4",
    provider=OpenAIProvider(base_url=base_url, api_key=api_key),
)

# This will trigger the model change observer
agent.model = new_model

Model changed from OpenAIModel() to OpenAIModel()


In [11]:
# Change Instructions
# This will trigger the instructions change observer
agent.instructions = "You are Dutch and respond in Dutch."

Instructions changed from You are German and respond in German. to You are Dutch and respond in Dutch.


In [12]:
await agent.run("Please greet Bob")

Agent starting with args: ('Please greet Bob',)
About to run tool: greet with args: {
  "name": "Bob"
}
Tool greet completed with result: Hello, Bob!
Agent finished


'Hello, Bob!'

In [13]:
agent.on.model

In [14]:
agent.on.instructions.value

'You are Dutch and respond in Dutch.'

In [15]:
# Watch both hook and value changes
def on_any_change(change):
    print(f"Trait {change.name} changed from {change.old} to {change.new}")

# Watch both hook and value changes
agent.on.model.observe(on_any_change, names=['hook', 'value'])


In [16]:
# Create a ReactiveTool
from pydantic_ai.hooks.tool import ReactiveTool

async def multiply(x: int) -> int:
    """Multiply a number by itself."""
    return x * x

reactive_tool = ReactiveTool(
    name="multiply", 
    function=multiply,
    description="Multiplies a number by itself"
)

# Watch tool state changes
def watch_state(change):
    print(f"Tool state changed from {change.old} to {change.new}")

reactive_tool.state.observe(watch_state, names='state')

# Add tool hooks
async def log_before(context):
    print(f"Tool about to run with args: {context['args']}")

async def log_after(context):
    print(f"Tool got result: {context['result']}")

async def log_error(context):
    print(f"Tool error occurred: {context['error']}")

reactive_tool.on.before = log_before
reactive_tool.on.after = log_after
reactive_tool.on.error = log_error

# Create a new agent with the reactive tool
reactive_agent = ReactiveAgent(
    model=model,
    tools=[reactive_tool],
    instructions="You are a math tutor."
)

# Add agent hooks
async def agent_before_tool(context):
    name = context['name']
    args = context['args']
    print(f"Agent about to run tool: {name} with args: {args}")

async def agent_after_tool(context):
    name = context['name']
    result = context['result']
    print(f"Agent tool {name} completed with result: {result}")

reactive_agent.on.before_any_tool = agent_before_tool
reactive_agent.on.after_any_tool = agent_after_tool

# Test agent run
await reactive_agent.run("What is 7 multiplied by itself?")


Agent about to run tool: multiply with args: {"x":7}
Agent tool multiply completed with result: 49


49